# SwiGLU FFN — Solution

Reference implementation for `01_swiglu_ffn_exercise.ipynb`.

## Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Solution 1 — `StandardFFN`

In [2]:
class StandardFFN(nn.Module):
    def __init__(self,d_model,d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = 4*d_model
         
        self.linear_1 = nn.Linear(d_model,d_ff)
        self.activation = nn.ReLU()
        self.linear_2 = nn.Linear(d_ff,d_model)

    def forward(self,x):
        out = self.activation(self.linear_1(x))

        return self.linear_2(out)

## Solution 2 — `GELUFFN`

In [3]:
class GELUFFN(nn.Module):
    def __init__(self,d_model,d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = 4*d_model
         
        self.linear_1 = nn.Linear(d_model,d_ff)
        self.activation = nn.GELU()
        self.linear_2 = nn.Linear(d_ff,d_model)

    def forward(self,x):
        out = self.activation(self.linear_1(x))

        return self.linear_2(out)

# Solution 3 - GLU

In [4]:
class GLUFFN(nn.Module):
    def __init__(self,d_model,d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = int(8*d_model/3)

        self.w_gate = nn.Linear(d_model,d_ff,bias=False)
        self.w_up = nn.Linear(d_model,d_ff,bias=False)
        self.w_down = nn.Linear(d_ff,d_model,bias=False)

    def forward(self,x):
        gate = F.sigmoid(self.w_gate(x))

        up_proj = self.w_up(x)

        out = gate * up_proj

        return self.w_down(out)

## Solution 4 — `SwiGLU`

In [5]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = int(8*d_model/3)

        self.w_gate = nn.Linear(d_model,d_ff,bias=False)
        self.w_up = nn.Linear(d_model,d_ff,bias=False)
        self.w_down = nn.Linear(d_ff,d_model,bias=False)

    def forward(self,x):
        gate = F.silu(self.w_gate(x))

        up_proj = self.w_up(x)

        out = gate * up_proj

        return self.w_down(out)

**Why does the gate matter?** <br>
A fixed activation (ReLU/GELU/SiLU) makes the same decision dimension-by-dimension regardless of the rest of the input. The gate path V(x) sees the FULL input and learns input-dependent, per-dimension scaling. This is strictly more expressive — the output at each dimension becomes a product of two learned functions of the input, not a single fixed nonlinearity. </br>

**Why 8/3 × d_model for hidden?** <br>Standard FFN has 2 matrices of size d×d_ff each → 2·d·(4d) = 8d². SwiGLU has 3 matrices of size d×d_ff → 3·d·d_ff. Setting them equal: d_ff = 8/3·d. So SwiGLU matches param count by shrinking the hidden dimension while adding an extra projection. Same compute, better quality.

## Run the tests

In [6]:
from tests import run_ffn_tests
run_ffn_tests(StandardFFN, GELUFFN, SwiGLU)

Running FFN variants...
  ✓ StandardFFN shape correct
  ✓ GELUFFN shape correct
  ✓ SwiGLU shape correct
  ✓ SwiGLU has 3 Linear layers
  ✓ Param counts roughly equal
  ✓ Gradients flow in all variants

  All 6 checks passed ✓



True